In [ ]:
# ==============================
# CELL 1 — Libraries
# ==============================

import ast
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

In [ ]:
# ==============================
# CELL 2 — Load Dataset
# ==============================

developers = pd.read_csv("../data/developers.csv")
repos = pd.read_csv("../data/repositories.csv")
edges = pd.read_csv("../data/edgelist.csv")

print("Developers shape:", developers.shape)
print("Repositories shape:", repos.shape)
print("Edges shape:", edges.shape)

print("\nDevelopers columns:")
print(developers.columns.tolist())

print("\nRepositories columns:")
print(repos.columns.tolist())

print("\nEdges columns:")
print(edges.columns.tolist())

In [ ]:
# ==============================
# CELL 3 — Dataset Explanation
# ==============================

summary_table = pd.DataFrame({
    "Dataset": ["developers.csv", "repositories.csv", "edgelist.csv"],
    "Rows": [len(developers), len(repos), len(edges)],
    "Columns": [developers.shape[1], repos.shape[1], edges.shape[1]],
    "Description": [
        "Contains developer profile and social information.",
        "Contains repository metadata, technical properties, and textual content.",
        "Contains developer-repository interaction relationships."
    ]
})

display(summary_table)

In [ ]:
# ==============================
# CELL 4 — Basic Dataset Analysis
# ==============================

# Interactions per developer
interactions_per_dev = edges.groupby("dev_id").size().sort_values(ascending=False)

plt.figure(figsize=(8,5))
sns.histplot(interactions_per_dev, bins=30)
plt.title("Distribution of Interactions per Developer")
plt.xlabel("Number of Interactions")
plt.ylabel("Frequency")
plt.show()

# Top languages from repositories
def safe_parse_list(x):
    if pd.isna(x):
        return []
    if isinstance(x, list):
        return x
    try:
        parsed = ast.literal_eval(x)
        if isinstance(parsed, list):
            return parsed
        return []
    except:
        return []

repos["languages_list"] = repos["languages"].apply(safe_parse_list)
repos["topics_list"] = repos["topics"].apply(safe_parse_list)

all_languages = [lang for sublist in repos["languages_list"] for lang in sublist]
lang_counts = pd.Series(all_languages).value_counts().head(10)

plt.figure(figsize=(10,5))
sns.barplot(x=lang_counts.values, y=lang_counts.index)
plt.title("Top 10 Programming Languages")
plt.xlabel("Count")
plt.ylabel("Language")
plt.show()

# Top topics
all_topics = [topic for sublist in repos["topics_list"] for topic in sublist]
if len(all_topics) > 0:
    topic_counts = pd.Series(all_topics).value_counts().head(10)

    plt.figure(figsize=(10,5))
    sns.barplot(x=topic_counts.values, y=topic_counts.index)
    plt.title("Top 10 Repository Topics")
    plt.xlabel("Count")
    plt.ylabel("Topic")
    plt.show()
else:
    print("No usable topics found in dataset.")

In [ ]:
# ==============================
# CELL 5 — Merge and Feature Engineering (FIXED FOR YOUR DATASET)
# ==============================

# Merge
data = edges.merge(developers, on="dev_id", how="left")
data = data.merge(repos, on="repo_id", how="left")

# Numeric developer columns
data["Followers"] = pd.to_numeric(data["Followers"], errors="coerce").fillna(0)
data["Following"] = pd.to_numeric(data["Following"], errors="coerce").fillna(0)
data["Public Repositories"] = pd.to_numeric(data["Public Repositories"], errors="coerce").fillna(0)
data["starredRepoCount"] = pd.to_numeric(data["starredRepoCount"], errors="coerce").fillna(0)
data["yearly_contributions"] = pd.to_numeric(data["yearly_contributions"], errors="coerce").fillna(0)

# Social score
data["social_score"] = (
    data["Followers"] +
    data["Following"] +
    data["Public Repositories"] +
    data["starredRepoCount"] +
    data["yearly_contributions"]
)

# Numeric repository columns
repo_numeric_cols = [
    "stargazers_count", "forks_count", "watching",
    "contributors_count", "commits_count",
    "open_issues_count", "size"
]

for col in repo_numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce").fillna(0)

data["repo_popularity_score"] = (
    data["stargazers_count"] +
    data["forks_count"] +
    data["watching"] +
    data["contributors_count"] +
    data["commits_count"]
)

# Text columns
data["description"] = data["description"].fillna("").astype(str)
data["readme"] = data["readme"].fillna("").astype(str)
data["topics"] = data["topics"].fillna("[]").astype(str)
data["languages"] = data["languages"].fillna("[]").astype(str)
data["Bio"] = data["Bio"].fillna("").astype(str)

# Text fields for modeling
data["repo_text"] = (
    data["description"] + " " +
    data["topics"] + " " +
    data["languages"]
)

data["dev_text"] = data["Bio"]

print("Merged data shape:", data.shape)
display(data[[
    "dev_id", "repo_id", "Followers", "Following",
    "Public Repositories", "starredRepoCount",
    "yearly_contributions", "social_score"
]].head())

In [ ]:
# ==============================
# CELL 6 — Technical-Interest Measurement Model
# ==============================

# Repo text TF-IDF
repo_vectorizer = TfidfVectorizer(max_features=500, stop_words="english")
repo_tfidf = repo_vectorizer.fit_transform(data["repo_text"])

# Developer text TF-IDF
dev_vectorizer = TfidfVectorizer(max_features=300, stop_words="english")
dev_tfidf = dev_vectorizer.fit_transform(data["dev_text"])

# Developer profile from repositories they interacted with
dev_profiles = {}

for dev in data["dev_id"].unique():
    idx = np.where(data["dev_id"].values == dev)[0]
    if len(idx) > 0:
        dev_profiles[dev] = np.asarray(repo_tfidf[idx].mean(axis=0))

technical_scores = []

for i, dev in enumerate(data["dev_id"]):
    dev_vec = dev_profiles[dev]
    repo_vec = repo_tfidf[i].toarray()
    score = cosine_similarity(dev_vec, repo_vec)[0][0]
    technical_scores.append(score)

data["technical_score"] = technical_scores

display(data[["dev_id", "repo_id", "technical_score"]].head())
print(data["technical_score"].describe())

In [ ]:
# ==============================
# CELL 7 — Social / Interaction Features (FIXED)
# ==============================

# Edge columns: fill missing values first
data["isForked"] = pd.to_numeric(data["isForked"], errors="coerce").fillna(0).astype(int)
data["isTopContributor"] = pd.to_numeric(data["isTopContributor"], errors="coerce").fillna(0).astype(int)

# Social interaction score
data["edge_social_signal"] = data["isForked"] + data["isTopContributor"]

# Final social score
data["social_score_final"] = data["social_score"] + data["edge_social_signal"]

display(data[[
    "dev_id", "repo_id", "isForked", "isTopContributor",
    "social_score", "edge_social_signal", "social_score_final"
]].head())

In [ ]:
# ==============================
# CELL 8 — Linear Combination Model
# ==============================

scaler = MinMaxScaler()

data["technical_norm"] = scaler.fit_transform(data[["technical_score"]])
data["social_norm"] = scaler.fit_transform(data[["social_score_final"]])
data["repo_popularity_norm"] = scaler.fit_transform(data[["repo_popularity_score"]])

# Weighted linear recommendation score
data["linear_score"] = (
    0.5 * data["technical_norm"] +
    0.3 * data["social_norm"] +
    0.2 * data["repo_popularity_norm"]
)

display(data[[
    "dev_id", "repo_id", "technical_norm", "social_norm", "repo_popularity_norm", "linear_score"
]].head())

In [ ]:
# ==============================
# CELL 9 — Prepare Data for Learning-to-Rank
# ==============================

positive_samples = data[[
    "dev_id", "repo_id", "technical_norm", "social_norm", "repo_popularity_norm"
]].copy()
positive_samples["label"] = 1

all_devs = data["dev_id"].unique()
all_repos = data["repo_id"].unique()
existing_pairs = set(zip(data["dev_id"], data["repo_id"]))

negative_rows = []
num_negative = len(positive_samples)

# For reproducibility
random.seed(42)

while len(negative_rows) < num_negative:
    dev = random.choice(all_devs)
    repo = random.choice(all_repos)

    if (dev, repo) not in existing_pairs:
        dev_rows = data[data["dev_id"] == dev]

        tech_val = dev_rows["technical_norm"].mean() if not dev_rows.empty else 0
        social_val = dev_rows["social_norm"].mean() if not dev_rows.empty else 0
        pop_val = data[data["repo_id"] == repo]["repo_popularity_norm"].mean()
        pop_val = 0 if pd.isna(pop_val) else pop_val

        negative_rows.append([dev, repo, tech_val, social_val, pop_val, 0])

negative_samples = pd.DataFrame(
    negative_rows,
    columns=[
        "dev_id", "repo_id", "technical_norm",
        "social_norm", "repo_popularity_norm", "label"
    ]
)

ranking_data = pd.concat([positive_samples, negative_samples], ignore_index=True)

print("Positive samples:", len(positive_samples))
print("Negative samples:", len(negative_samples))
print("Ranking data shape:", ranking_data.shape)

ranking_data.head()

In [ ]:
# ==============================
# CELL 10 — Learning-to-Rank Model
# ==============================

features = ["technical_norm", "social_norm", "repo_popularity_norm"]

X = ranking_data[features]
y = ranking_data["label"]

ltr_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

ltr_model.fit(X, y)

ranking_data["ltr_score"] = ltr_model.predict_proba(X)[:, 1]

print(classification_report(y, ltr_model.predict(X)))
print("ROC-AUC:", roc_auc_score(y, ranking_data["ltr_score"]))

ranking_data[["dev_id", "repo_id", "ltr_score", "label"]].head()

In [ ]:
# ==============================
# CELL 11 — LTR Feature Importance
# ==============================

importance_df = pd.DataFrame({
    "Feature": features,
    "Importance": ltr_model.feature_importances_
}).sort_values("Importance", ascending=False)

display(importance_df)

plt.figure(figsize=(8,4))
sns.barplot(data=importance_df, x="Importance", y="Feature")
plt.title("Feature Importance in Learning-to-Rank Model")
plt.show()

In [ ]:
# ==============================
# CELL 12 — Comparison Table
# ==============================

comparison_df = ranking_data.merge(
    data[["dev_id", "repo_id", "linear_score"]],
    on=["dev_id", "repo_id"],
    how="left"
)

comparison_table = pd.DataFrame({
    "Metric": ["Mean", "Std", "Min", "Max"],
    "Linear Score": [
        comparison_df["linear_score"].mean(),
        comparison_df["linear_score"].std(),
        comparison_df["linear_score"].min(),
        comparison_df["linear_score"].max(),
    ],
    "LTR Score": [
        comparison_df["ltr_score"].mean(),
        comparison_df["ltr_score"].std(),
        comparison_df["ltr_score"].min(),
        comparison_df["ltr_score"].max(),
    ]
})

display(comparison_table)

In [ ]:
# ==============================
# CELL 13 — Score Distribution Plot
# ==============================

plt.figure(figsize=(8,5))
sns.histplot(comparison_df["linear_score"].dropna(), label="Linear Score", kde=True, alpha=0.5)
sns.histplot(comparison_df["ltr_score"].dropna(), label="LTR Score", kde=True, alpha=0.5)

plt.title("Distribution of Linear and LTR Scores")
plt.xlabel("Score")
plt.ylabel("Frequency")
plt.legend()
plt.show()

In [ ]:
# ==============================
# CELL 14 — Recommendation Function
# ==============================

def recommend_for_developer(dev_id, top_n=5):
    # Linear recommendations
    linear_rec = data[data["dev_id"] == dev_id][["repo_id", "linear_score"]].copy()
    linear_rec = linear_rec.sort_values("linear_score", ascending=False).head(top_n)

    # LTR recommendations
    ltr_rec = ranking_data[ranking_data["dev_id"] == dev_id][["repo_id", "ltr_score", "label"]].copy()
    ltr_rec = ltr_rec.sort_values("ltr_score", ascending=False).head(top_n)

    return linear_rec, ltr_rec

sample_dev = data["dev_id"].iloc[0]

linear_rec, ltr_rec = recommend_for_developer(sample_dev, top_n=5)

print(f"Sample developer id: {sample_dev}")

print("\n=== Linear Combination Recommendations ===")
display(linear_rec)

print("\n=== Learning-to-Rank Recommendations ===")
display(ltr_rec)

In [ ]:
# ==============================
# CELL 15 — Recommendation Results with Repo Names
# ==============================

repo_names = repos[["repo_id", "repo_name", "owner_username", "stargazers_count", "languages", "topics"]].copy()

linear_named = linear_rec.merge(repo_names, on="repo_id", how="left")
ltr_named = ltr_rec.merge(repo_names, on="repo_id", how="left")

print("=== Linear Recommendations with Repository Names ===")
display(linear_named)

print("=== LTR Recommendations with Repository Names ===")
display(ltr_named)